# 08. 정류장 단위 재추정 — 250m 격자를 쓰지 않아도 같은 결과가 나오나

**문제**: 지금까지는 250m 격자 + 공급구조 유지 셀 132개만 썼다. 격자를 어떻게 자르느냐, 어떤 셀을 고르느냐에 결과가 달려 있을 수 있다.

**방법**
- 단위: **정류장 × 노선** (전체 24개 노선의 모든 경기 승차 정류장. 격자·안정 셀 조건 없음)
- 거리: 정류장 좌표 → 최근접 GTX-A역 **직선거리(km)** (보행시간은 격자 중심점에만 있어서 직선거리를 씀. 셀 단위에서 보행시간과 직선거리 상관 0.99)
- 노선 고정효과 + 정류장 군집 표준오차
- 표본 두 가지
  - A: 2024년에 승차가 있던 모든 정류장×노선
  - B: 2025년에도 그 노선 승차 기록이 있는 쌍만 (노선이 정류장을 더 이상 안 지나는 경우를 제거한 **보수적** 표본. 다만 2025년 0명인 쌍도 빠짐)

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # 정리본 최상위의 config.py

from config import *
setup_notebook()

import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy.stats import norm

data = load_model_data()  # 1_받은코드_정리/output/model_dataset.csv
X_DUMMY = ["walk_20", "walk_20_60", "ic_access", "competing_share", "pop_20_50", "buildings"]

In [2]:
trips = load_trips()
coords = trips.groupby("정류장")[["위도", "경도"]].first()
coords["GTX 직선거리(km)"] = nearest_gtx_km(coords.위도, coords.경도)

pairs = (trips.pivot_table(index=["정류장", "노선"], columns="연도", values="승차인원", aggfunc="sum", fill_value=0)
         .reset_index().rename(columns={2024: "y2024", 2025: "y2025"}))
pairs = pairs.merge(coords[["GTX 직선거리(km)"]], left_on="정류장", right_index=True)
served_2025 = set(map(tuple, trips.loc[trips.연도 == 2025, ["정류장", "노선"]].drop_duplicates().values))
pairs["2025 운행 확인"] = [(s, r) in served_2025 for s, r in zip(pairs.정류장, pairs.노선)]
pairs = pairs[pairs.y2024 > 0].reset_index(drop=True)
samples = {"A: 2024 승차 있는 전체": pairs, "B: 2025에도 운행 확인": pairs[pairs["2025 운행 확인"]].reset_index(drop=True)}
for name, frame in samples.items():
    print(f"{name}: 정류장×노선 {len(frame)}쌍, 정류장 {frame.정류장.nunique()}개")

A: 2024 승차 있는 전체: 정류장×노선 363쌍, 정류장 228개
B: 2025에도 운행 확인: 정류장×노선 292쌍, 정류장 174개


In [3]:
rows = []
for name, frame in samples.items():
    route_dummies = pd.get_dummies(frame.노선, prefix="노선", drop_first=True, dtype=float)
    for cutoff in (1.0, 1.25, 1.5, 2.0, 3.0):
        near = (frame["GTX 직선거리(km)"] <= cutoff).astype(float).rename("near")
        for fe in (False, True):
            X = pd.concat([near, route_dummies], axis=1) if fe else near.to_frame()
            result = poisson_rate(frame.y2025, frame.y2024, X, cluster=frame.정류장)
            rows.append({"표본": name, "기준(km)": cutoff, "기준 이내 정류장": frame.loc[near == 1, "정류장"].nunique(),
                         "노선 FE": "포함" if fe else "없음",
                         "변화율 차이(%)": (np.exp(result.params["near"]) - 1) * 100, "p": result.pvalues["near"]})
result_table = pd.DataFrame(rows).pivot_table(index=["표본", "기준(km)", "기준 이내 정류장"], columns="노선 FE", values=["변화율 차이(%)", "p"])
result_table.round(4)

p         변화율 차이(%)         
노선 FE                                  없음      포함        없음       포함
표본               기준(km) 기준 이내 정류장                                   
A: 2024 승차 있는 전체 1.00   9          0.3905  0.0194  -17.4290 -45.4547
                 1.25   19         0.0296  0.0003  -25.8779 -40.1408
                 1.50   24         0.6824  0.0002   -6.8506 -27.0402
                 2.00   50         0.7595  0.0976   -3.3343 -11.9627
                 3.00   95         0.7002  0.1223   -3.8289 -11.9381
B: 2025에도 운행 확인  1.00   5          0.3752  0.0256  -17.8565 -44.4493
                 1.25   15         0.0164  0.0003  -27.8996 -40.2271
                 1.50   19         0.5376  0.0001  -10.0869 -27.2900
                 2.00   41         0.4456  0.0655   -8.0037 -13.2149
                 3.00   82         0.0680  0.0084  -15.0775 -18.7765

In [4]:
# 연속형: log(직선거리) + 노선 고정효과
for name, frame in samples.items():
    X = pd.concat([np.log(frame[["GTX 직선거리(km)"]]).rename(columns=lambda c: "log_dist"),
                   pd.get_dummies(frame.노선, prefix="노선", drop_first=True, dtype=float)], axis=1)
    result = poisson_rate(frame.y2025, frame.y2024, X, cluster=frame.정류장)
    print(f"{name}: 거리 2배 → 변화율 {(2 ** result.params['log_dist'] - 1) * 100:+.1f}%  (β={result.params['log_dist']:.3f}, p={result.pvalues['log_dist']:.4f})")

A: 2024 승차 있는 전체: 거리 2배 → 변화율 +11.7%  (β=0.160, p=0.0336)
B: 2025에도 운행 확인: 거리 2배 → 변화율 +14.9%  (β=0.201, p=0.0191)


## 결론
- 250m 격자와 안정 셀 조건을 버리고 **정류장 단위(직선거리)**로 다시 해도, **노선 고정효과를 넣으면** 같은 결론이 나온다:
  - 1~1.5km 이내 정류장의 변화율이 −27 ~ −45% 낮다 (p < 0.03).
  - 거리가 2배가 되면 변화율이 +12 ~ +15% 높다 (p < 0.04).
- **노선 고정효과가 없으면 대부분 유의하지 않다.** 노선 구성을 통제하는 것이 핵심이라는 07의 결론과 같다.
- 효과는 약 1.5km 이내에서 뚜렷하고, 2km 이상 기준에서는 약해진다. 셀 단위 보행 20~25분과 비슷한 범위다.
- 한계: 직선거리를 썼고(보행시간은 격자 중심점에만 있음), 표본 A에는 2025년에 노선이 정류장을 더 이상 지나지 않는 경우도 섞여 있을 수 있다. 그래서 보수적인 표본 B도 같이 보고한다(결과 동일).